In [1]:
import numpy as np
import pandas as pd
import pyswarms as ps
import joblib
import pygad
import neat

from pureples.shared import Substrate
from pureples.es_hyperneat.es_hyperneat import ESNetwork
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, classification_report

In [ ]:
big_data = pd.read_csv("big_data.csv").drop(columns=["Unnamed: 0"])
small_data = pd.read_csv("small_data.csv").drop(columns=["Unnamed: 0"])

X_big = big_data.drop(columns=["collision"])
y_big = big_data["collision"]

X_small = small_data.drop(columns=["collision"])
y_small = small_data["collision"]

smote = SMOTE(random_state=42)

X_big_train, X_big_test, y_big_train, y_big_test = train_test_split(X_big, y_big, test_size=0.2, stratify=y_big, random_state=42)
X_big_train_b, y_big_train_b = smote.fit_resample(X_big_train, y_big_train)

X_small_train, X_small_test, y_small_train, y_small_test = train_test_split(X_small, y_small, test_size=0.2, stratify=y_small, random_state=42)
X_small_train_b, y_small_train_b = smote.fit_resample(X_small_train, y_small_train)

In [3]:
old_logit = joblib.load('D:/Python/OmGTU/Practicum/LAB_2/logit.joblib')
old_logit.fit(X_big_train_b, y_big_train_b)

LogisticRegression(C=5.565179891307081, fit_intercept=False, max_iter=3000)

In [4]:
old_cart = joblib.load('D:/Python/OmGTU/Practicum/LAB_2/cart.joblib')
old_cart.fit(X_big_train_b, y_big_train_b)

DecisionTreeClassifier(ccp_alpha=0.03682700040880698, max_depth=14,
                       max_leaf_nodes=83,
                       min_impurity_decrease=0.04740650391151835,
                       min_samples_leaf=9, min_samples_split=7,
                       min_weight_fraction_leaf=0.08058917660642025)

# Задание 1. Классический генетический алгоритм

## Logit

In [5]:
MODEL_LOGIT = {
    "model": LogisticRegression,
    "params": {
        "C": (0.001, 100),
        "penalty": ["l1", "l2", "elasticnet", None],
        "solver": ["liblinear", "saga", "newton-cg", "lbfgs"],
        "l1_ratio": (0, 1),
        "class_weight": [None, "balanced"]
    }
}

def gene_to_params_logit(genes):
    return {
        "C": 10 ** genes[0],
        "penalty": MODEL_LOGIT["params"]["penalty"][int(genes[1])],
        "solver": MODEL_LOGIT["params"]["solver"][int(genes[2])]
    }

def fitness_func_logit(ga_instance, solution, solution_idx):
    params = gene_to_params_logit(solution)
    
    try:
        model = LogisticRegression(**params, max_iter=5000)
        model.fit(X_big_train_b, y_big_train_b)
        y_pred = model.predict(X_big_test)
        
        if len(np.unique(y_pred)) < 2:
            return 0
            
        f1 = f1_score(y_big_test, y_pred, zero_division=0)
    except Exception as e:
        f1 = 0
    return f1

ga_logit = pygad.GA(
    num_generations=50,
    num_parents_mating=10,
    fitness_func=fitness_func_logit,
    sol_per_pop=30,
    num_genes=3,
    gene_space=[
        {"low": np.log10(0.001), "high": np.log10(100)},
        [0, 1],
        [0, 1]
    ],
    mutation_probability=[0.05, 0.2],
    mutation_type="adaptive",
    gene_type=float,
    stop_criteria="saturate_10"
)

ga_logit.run()

best_logit_params = gene_to_params_logit(ga_logit.best_solution()[0])
best_logit_ga = LogisticRegression(**best_logit_params, max_iter=5000)
best_logit_ga.fit(X_big_train_b, y_big_train_b)
joblib.dump(best_logit_ga, "best_logit_ga.joblib")

d:\Python\OmGTU\venv\Lib\site-packages\pygad\pygad.py:698: UserWarning: The first element in the 'mutation_probability' parameter is 0.05 which is smaller than the second element 0.2. This means the mutation rate for the high-quality solutions is higher than the mutation rate of the low-quality ones. This causes high disruption in the high qualitiy solutions while making little changes in the low quality solutions. Please make the first element higher than the second element.
  warnings.warn(f"The first element in the 'mutation_probability' parameter is {mutation_probability[0]} which is smaller than the second element {mutation_probability[1]}. This means the mutation rate for the high-quality solutions is higher than the mutation rate of the low-quality ones. This causes high disruption in the high qualitiy solutions while making little changes in the low quality solutions. Please make the first element higher than the second element.")


['best_logit_ga.joblib']

## CART

In [6]:
MODEL_CART = {
    "model": DecisionTreeClassifier,
    "params": {
        "max_depth": (3, 20),
        "min_samples_split": (2, 10),
        "criterion": ["gini", "entropy"]
    }
}

def gene_to_params_cart(genes):
    return {
        "max_depth": int(genes[0]),
        "min_samples_split": int(genes[1]),
        "criterion": MODEL_CART["params"]["criterion"][int(genes[2])]
    }

def fitness_func_cart(ga_instance, solution, solution_idx):
    params = gene_to_params_cart(solution)
    
    try:
        model = DecisionTreeClassifier(**params)
        model.fit(X_big_train_b, y_big_train_b)
        y_pred = model.predict(X_big_test)
        
        if len(np.unique(y_pred)) < 2:
            return 0
            
        f1 = f1_score(y_big_test, y_pred, zero_division=0)
    except Exception as e:
        f1 = 0
    return f1

ga_cart = pygad.GA(
    num_generations=20,
    num_parents_mating=5,
    fitness_func=fitness_func_cart,
    sol_per_pop=15,
    num_genes=3,
    gene_space=[
        {"low": 3, "high": 20},
        {"low": 2, "high": 10},
        [0, 1]
    ],
    mutation_probability=0.15,
    gene_type=float
)

ga_cart.run()

best_cart_params = gene_to_params_cart(ga_cart.best_solution()[0])
best_cart_ga = DecisionTreeClassifier(**best_cart_params)
best_cart_ga.fit(X_big_train, y_big_train)
joblib.dump(best_cart_ga, "best_cart_ga.joblib")

['best_cart_ga.joblib']

# Задание 2. Алгоритм роя частиц

## Logit

In [7]:
def f1_swarm_logit(params):
    f1_scores = []
    for param_set in params:
        try:
            model = LogisticRegression(
                C=10 ** param_set[0],
                penalty=["l1", "l2"][int(param_set[1])],
                solver=["liblinear", "saga"][int(param_set[2])],
                max_iter=1000
            )
            model.fit(X_big_train_b, y_big_train_b)
            y_pred = model.predict(X_big_test)
            f1 = f1_score(y_big_test, y_pred, zero_division=0)
        except:
            f1 = 0
        f1_scores.append(f1)
    return -np.array(f1_scores)

options = {"c1": 0.5, "c2": 0.3, "w": 0.9}
optimizer_logit = ps.single.GlobalBestPSO(
    n_particles=20,
    dimensions=3,
    options=options,
    bounds=(
        [np.log10(0.001), 0, 0],
        [np.log10(100), 1, 1]
    )
)

best_cost_logit, best_pos_logit = optimizer_logit.optimize(f1_swarm_logit, iters=20)

best_logit_sw = LogisticRegression(
    C=10 ** best_pos_logit[0],
    penalty=["l1", "l2"][int(best_pos_logit[1])],
    solver=["liblinear", "saga"][int(best_pos_logit[2])],
    max_iter=5000
)
best_logit_sw.fit(X_big_train_b, y_big_train_b)
joblib.dump(best_logit_sw, "best_logit_pso.joblib")

2025-03-29 01:14:42,282 - pyswarms.single.global_best - INFO - Optimize for 20 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|20/20, best_cost=-0.438
2025-03-29 01:15:14,088 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.4380952380952381, best pos: [-2.53173846  0.24138763  0.41747612]


['best_logit_pso.joblib']

## CART

In [8]:
def f1_swarm_cart(params):
    f1_scores = []
    for param_set in params:
        try:
            model = DecisionTreeClassifier(
                max_depth=int(param_set[0]),
                min_samples_split=int(param_set[1]),
                criterion=["gini", "entropy"][int(param_set[2])],
                random_state=42
            )
            model.fit(X_big_train_b, y_big_train_b)
            y_pred = model.predict(X_big_test)
            f1 = f1_score(y_big_test, y_pred, zero_division=0)
        except:
            f1 = 0
        f1_scores.append(f1)
    return -np.array(f1_scores)

options = {"c1": 0.5, "c2": 0.3, "w": 0.9}
optimizer_cart = ps.single.GlobalBestPSO(
    n_particles=20,
    dimensions=3,
    options=options,
    bounds=(
        [3, 2, 0],
        [20, 10, 1]
    )
)

best_cost_cart, best_pos_cart = optimizer_cart.optimize(f1_swarm_cart, iters=20)

best_cart_sw = DecisionTreeClassifier(
    max_depth=int(best_pos_cart[0]),
    min_samples_split=int(best_pos_cart[1]),
    criterion=["gini", "entropy"][int(best_pos_cart[2])],
    random_state=42
)
best_cart_sw.fit(X_big_train_b, y_big_train_b)
joblib.dump(best_cart_sw, "best_cart_pso.joblib")

2025-03-29 01:15:14,129 - pyswarms.single.global_best - INFO - Optimize for 20 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|20/20, best_cost=-0.398
2025-03-29 01:15:20,582 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.39790575916230364, best pos: [15.04971601  4.07072095  0.31926448]


['best_cart_pso.joblib']

In [9]:
y_pred_logit_old = old_logit.predict(X_big_test)
y_pred_logit_ga = best_logit_ga.predict(X_big_test)
y_pred_logit_sw = best_logit_sw.predict(X_big_test)

print(classification_report(y_big_test, y_pred_logit_old))
print(classification_report(y_big_test, y_pred_logit_ga))
print(classification_report(y_big_test, y_pred_logit_sw))

              precision    recall  f1-score   support

           0       0.72      0.88      0.79       209
           1       0.43      0.21      0.28        91

    accuracy                           0.68       300
   macro avg       0.58      0.54      0.54       300
weighted avg       0.63      0.68      0.64       300

              precision    recall  f1-score   support

           0       0.74      0.53      0.61       209
           1       0.34      0.57      0.43        91

    accuracy                           0.54       300
   macro avg       0.54      0.55      0.52       300
weighted avg       0.62      0.54      0.56       300

              precision    recall  f1-score   support

           0       0.75      0.65      0.70       209
           1       0.39      0.51      0.44        91

    accuracy                           0.61       300
   macro avg       0.57      0.58      0.57       300
weighted avg       0.64      0.61      0.62       300



In [10]:
y_pred_cart_old = old_cart.predict(X_big_test)
y_pred_cart_ga = best_cart_ga.predict(X_big_test)
y_pred_cart_sw = best_cart_sw.predict(X_big_test)

print(classification_report(y_big_test, y_pred_cart_old))
print(classification_report(y_big_test, y_pred_cart_ga))
print(classification_report(y_big_test, y_pred_cart_sw))

              precision    recall  f1-score   support

           0       0.70      1.00      0.82       209
           1       0.00      0.00      0.00        91

    accuracy                           0.70       300
   macro avg       0.35      0.50      0.41       300
weighted avg       0.49      0.70      0.57       300

              precision    recall  f1-score   support

           0       0.72      0.77      0.74       209
           1       0.37      0.32      0.34        91

    accuracy                           0.63       300
   macro avg       0.55      0.54      0.54       300
weighted avg       0.61      0.63      0.62       300

              precision    recall  f1-score   support

           0       0.73      0.70      0.72       209
           1       0.38      0.42      0.40        91

    accuracy                           0.62       300
   macro avg       0.56      0.56      0.56       300
weighted avg       0.63      0.62      0.62       300



d:\Python\OmGTU\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Python\OmGTU\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Python\OmGTU\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Задание 3. Алгоритм NEAT

In [12]:
data = pd.read_csv("small_data.csv", index_col=0)
X_small = data.drop("collision", axis=1).values
y_small = data["collision"].values

print("Форма данных X_small:", X_small.shape)

data = pd.read_csv("big_data.csv", index_col=0)
X_big = data.drop("collision", axis=1).values
y_big = data["collision"].values

print("Форма данных X_big:", X_big.shape)

Форма данных X_small: (70, 10)
Форма данных X_big: (1500, 24)


## small

In [13]:
def eval_genomes(genomes, config):
    for genome_id, genome in genomes:
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        predictions = []
        for xi in X_small:
            output = net.activate(xi)
            predictions.append(round(output[0]))

        genome.fitness = f1_score(y_small, predictions, average='weighted')

config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                    neat.DefaultSpeciesSet, neat.DefaultStagnation,
                    'small_data.cfg')

population = neat.Population(config)
population.add_reporter(neat.StdOutReporter(True))
population.add_reporter(neat.StatisticsReporter())

winner = population.run(eval_genomes, 20)

winner_net_small = neat.nn.FeedForwardNetwork.create(winner, config)
predictions = [round(winner_net_small.activate(x)[0]) for x in X_small]


 ****** Running generation 0 ****** 

Population's average fitness: 0.52382 stdev: 0.13232
Best fitness: 0.71917 - size: (1, 10) - species 1 - id 149
Average adjusted fitness: 0.443
Mean genetic distance 0.858, standard deviation 0.310
Population of 150 members in 1 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    0   150      0.7    0.443     0
Total extinctions: 0
Generation time: 0.687 sec

 ****** Running generation 1 ****** 

Population's average fitness: 0.62234 stdev: 0.07717
Best fitness: 0.74168 - size: (1, 9) - species 1 - id 257
Average adjusted fitness: 0.389
Mean genetic distance 1.015, standard deviation 0.332
Population of 150 members in 1 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    1   150      0.7    0.389     0
Total extinctions: 0
Generation time: 0.678 sec (0.682 average)

 ****** Running generation 2 ****** 

Population's average fitness: 0.62665 stdev:

In [14]:
joblib.dump(winner_net_small, 'neat_small.joblib')

['neat_small.joblib']

In [15]:
print(classification_report(y_small, predictions))
print("Predictions:", predictions)
print("Actual:     ", y_small.tolist())

              precision    recall  f1-score   support

           0       0.83      0.92      0.88        53
           1       0.64      0.41      0.50        17

    accuracy                           0.80        70
   macro avg       0.73      0.67      0.69        70
weighted avg       0.78      0.80      0.78        70

Predictions: [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Actual:      [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]


## big

In [16]:
def eval_genomes(genomes, config):
    for genome_id, genome in genomes:
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        predictions = []
        for xi in X_big:
            output = net.activate(xi)
            predictions.append(round(output[0]))
        
        genome.fitness = f1_score(y_big, predictions, average='weighted')

config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                    neat.DefaultSpeciesSet, neat.DefaultStagnation,
                    'big_data.cfg')

population = neat.Population(config)
population.add_reporter(neat.StdOutReporter(True))
population.add_reporter(neat.StatisticsReporter())

winner = population.run(eval_genomes, 50)

winner_net_big = neat.nn.FeedForwardNetwork.create(winner, config)
predictions = [round(winner_net_big.activate(x)[0]) for x in X_big]


 ****** Running generation 0 ****** 

Population's average fitness: 0.42006 stdev: 0.19041
Best fitness: 0.61162 - size: (1, 17) - species 1 - id 120
Average adjusted fitness: 0.280
Mean genetic distance 1.235, standard deviation 0.293
Population of 200 members in 1 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    0   200      0.6    0.280     0
Total extinctions: 0
Generation time: 3.716 sec

 ****** Running generation 1 ****** 

Population's average fitness: 0.45395 stdev: 0.19034
Best fitness: 0.61480 - size: (2, 18) - species 1 - id 384
Average adjusted fitness: 0.454
Mean genetic distance 1.366, standard deviation 0.306
Population of 200 members in 1 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    1   200      0.6    0.454     0
Total extinctions: 0
Generation time: 3.608 sec (3.662 average)

 ****** Running generation 2 ****** 

Population's average fitness: 0.44952 stdev

In [17]:
joblib.dump(winner_net_big, 'neat_big.joblib')

['neat_big.joblib']

In [18]:
print(classification_report(y_big, predictions))
print("Predictions:", predictions)
print("Actual:     ", y_big.tolist())

              precision    recall  f1-score   support

           0       0.72      0.84      0.78      1046
           1       0.41      0.26      0.32       454

    accuracy                           0.66      1500
   macro avg       0.57      0.55      0.55      1500
weighted avg       0.63      0.66      0.64      1500

Predictions: [1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

# Задание 4. Алгоритм ES-HyperNEAT

## small

In [19]:
input_coordinates = [(-1.0, i*0.2) for i in range(10)]
output_coordinates = [(1.0, 0.0)]
substrate = Substrate(input_coordinates, output_coordinates)

es_params = {
    "initial_depth": 0,
    "max_depth": 2,
    "variance_threshold": 0.03,
    "band_threshold": 0.3,
    "iteration_level": 1,
    "division_threshold": 0.5,
    "max_weight": 5.0,
    "activation": "sigmoid"
}

def eval_genomes(genomes, config):
    for genome_id, genome in genomes:

        cppn = neat.nn.FeedForwardNetwork.create(genome, config)
        
        es_network = ESNetwork(substrate, cppn, es_params)
        
        net = es_network.create_phenotype_network()

        predictions = [round(net.activate(x)[0]) for x in X_small]
        genome.fitness = f1_score(y_small, predictions, average='weighted')

config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                     neat.DefaultSpeciesSet, neat.DefaultStagnation,
                     'small_data_upd.cfg')

population = neat.Population(config)
population.add_reporter(neat.StdOutReporter(True))
population.add_reporter(neat.StatisticsReporter())
winner = population.run(eval_genomes, 20)

winner_cppn = neat.nn.FeedForwardNetwork.create(winner, config)
final_es_net_small = ESNetwork(substrate, winner_cppn, es_params).create_phenotype_network()
predictions = [round(final_es_net_small.activate(x)[0]) for x in X_small]


 ****** Running generation 0 ****** 

Population's average fitness: 0.65250 stdev: 0.00000
Best fitness: 0.65250 - size: (1, 5) - species 1 - id 1
Average adjusted fitness: 0.000
Mean genetic distance 1.027, standard deviation 0.375
Population of 150 members in 1 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    0   150      0.7    0.000     0
Total extinctions: 0
Generation time: 1.611 sec

 ****** Running generation 1 ****** 

Population's average fitness: 0.65250 stdev: 0.00000
Best fitness: 0.65250 - size: (1, 5) - species 1 - id 1
Average adjusted fitness: 0.000
Mean genetic distance 1.239, standard deviation 0.411
Population of 150 members in 1 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    1   150      0.7    0.000     1
Total extinctions: 0
Generation time: 1.426 sec (1.518 average)

 ****** Running generation 2 ****** 

Population's average fitness: 0.65250 stdev: 0.00

In [20]:
joblib.dump(final_es_net_small, 'es_neat_small.joblib')

['es_neat_small.joblib']

In [21]:
print(classification_report(y_small, predictions))
print("Predictions:", predictions)
print("Actual:     ", y_small.tolist())

              precision    recall  f1-score   support

           0       0.76      1.00      0.86        53
           1       0.00      0.00      0.00        17

    accuracy                           0.76        70
   macro avg       0.38      0.50      0.43        70
weighted avg       0.57      0.76      0.65        70

Predictions: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Actual:      [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]


d:\Python\OmGTU\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Python\OmGTU\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Python\OmGTU\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## big

In [25]:
input_coordinates = [(x, y) for x in np.linspace(-1, 1, 6) 
                    for y in np.linspace(-0.8, 0.8, 4)][:24]
output_coordinates = [(1.0, 0.0)]
substrate = Substrate(input_coordinates, output_coordinates)

es_params = {
    "initial_depth": 1,
    "max_depth": 3,
    "variance_threshold": 0.05,
    "band_threshold": 0.4,
    "iteration_level": 2,
    "division_threshold": 0.6,
    "max_weight": 5.0,
    "activation": "sigmoid"
}

def eval_genomes(genomes, config):
    for genome_id, genome in genomes:
        cppn = neat.nn.FeedForwardNetwork.create(genome, config)
        es_net = ESNetwork(substrate, cppn, es_params)
        phenotype_net = es_net.create_phenotype_network()

        input_order = [tuple(coord) for coord in substrate.input_coordinates]
        
        predictions = []
        for xi in X_big:

            inputs = [xi[i] for i in range(len(input_order))]
            output = phenotype_net.activate(inputs)[0]
            predictions.append(round(output))
        
        genome.fitness = f1_score(y_big, predictions, average='weighted')

config = neat.Config(
    neat.DefaultGenome,
    neat.DefaultReproduction,
    neat.DefaultSpeciesSet,
    neat.DefaultStagnation,
    'big_data_upd.cfg'
)

population = neat.Population(config)
population.add_reporter(neat.StdOutReporter(True))
population.add_reporter(neat.StatisticsReporter())
winner = population.run(eval_genomes, 50)

winner_cppn = neat.nn.FeedForwardNetwork.create(winner, config)
final_es_net_big = ESNetwork(substrate, winner_cppn, es_params).create_phenotype_network()
predictions = [round(final_es_net_big.activate(x)[0]) for x in X_big]


 ****** Running generation 0 ****** 

Population's average fitness: 0.47133 stdev: 0.18341
Best fitness: 0.57299 - size: (1, 4) - species 1 - id 3
Average adjusted fitness: 0.331
Mean genetic distance 1.096, standard deviation 0.377
Population of 200 members in 1 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    0   200      0.6    0.331     0
Total extinctions: 0
Generation time: 19.458 sec

 ****** Running generation 1 ****** 

Population's average fitness: 0.49296 stdev: 0.16797
Best fitness: 0.57299 - size: (1, 4) - species 1 - id 3
Average adjusted fitness: 0.353
Mean genetic distance 1.351, standard deviation 0.380
Population of 200 members in 1 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    1   200      0.6    0.353     1
Total extinctions: 0
Generation time: 17.126 sec (18.292 average)

 ****** Running generation 2 ****** 

Population's average fitness: 0.50622 stdev: 0

In [26]:
joblib.dump(final_es_net_big, 'es_neat_big.joblib')

['es_neat_big.joblib']

In [28]:
print(classification_report(y_big, predictions))
print("Predictions:", predictions)
print("Actual:     ", y_big.tolist())

              precision    recall  f1-score   support

           0       0.71      0.83      0.77      1046
           1       0.35      0.21      0.26       454

    accuracy                           0.64      1500
   macro avg       0.53      0.52      0.51      1500
weighted avg       0.60      0.64      0.61      1500

Predictions: [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,